In [2]:
import polars as pl
from pathlib import Path

In [4]:
info_path = Path("/home/ubuntu/patent_similarity_new/data/stkcd_info.csv")
check_cols = ["province", "city", "Ind"]

target_stkcd = None


def clean_text_col(name: str) -> pl.Expr:
    cleaned = pl.col(name).cast(pl.Utf8).str.strip_chars()
    return (
        pl.when(pl.col(name).is_null() | (cleaned == ""))
        .then(pl.lit(None, dtype=pl.Utf8))
        .otherwise(cleaned)
        .alias(name)
    )


stkcd_info = (
    pl.read_csv(info_path)
    .with_columns(
        pl.col("stkcd").cast(pl.Utf8).str.zfill(6),
        pl.col("year").cast(pl.Int64, strict=False),
        *[clean_text_col(c) for c in check_cols],
    )
)

scan_base = (
    stkcd_info
    if target_stkcd is None
    else stkcd_info.filter(pl.col("stkcd") == str(target_stkcd).zfill(6))
)

stkcd_changes = (
    scan_base.group_by("stkcd")
    .agg(
        pl.col("year").min().alias("first_year"),
        pl.col("year").max().alias("last_year"),
        pl.len().alias("n_rows"),
        *[pl.col(c).drop_nulls().n_unique().alias(f"n_{c}") for c in check_cols],
        *[
            pl.col(c).drop_nulls().unique().sort().alias(f"{c}_values")
            for c in check_cols
        ],
    )
    .with_columns(
        pl.any_horizontal([pl.col(f"n_{c}") > 1 for c in check_cols]).alias(
            "has_different_province_city_ind"
        )
    )
    .filter(pl.col("has_different_province_city_ind"))
    .sort("stkcd")
)

stkcd_change_details = (
    stkcd_info.join(stkcd_changes.select("stkcd"), on="stkcd", how="semi")
    .select("stkcd", "year", *check_cols)
    .unique()
    .sort(["stkcd", "year"])
)

In [5]:
# Check whether the same city is linked to different province names.
city_province_base = stkcd_info.filter(
    pl.col("city").is_not_null() & pl.col("province").is_not_null()
)

city_province_conflicts = (
    city_province_base.group_by("city")
    .agg(
        pl.col("province").n_unique().alias("n_province"),
        pl.col("province").unique().sort().alias("province_values"),
        pl.len().alias("n_rows"),
        pl.col("stkcd").n_unique().alias("n_stkcd"),
        pl.col("year").min().alias("first_year"),
        pl.col("year").max().alias("last_year"),
    )
    .filter(pl.col("n_province") > 1)
    .sort(["n_province", "city"], descending=[True, False])
)

city_province_conflict_details = (
    city_province_base.join(
        city_province_conflicts.select("city"), on="city", how="semi"
    )
    .select("city", "province", "stkcd", "year", "Ind")
    .unique()
    .sort(["city", "province", "stkcd", "year"])
)

print(f"{city_province_conflicts.height} cities have multiple province names.")
city_province_conflicts

1 cities have multiple province names.


city,n_province,province_values,n_rows,n_stkcd,first_year,last_year
str,u32,list[str],u32,u32,i64,i64
"""香港特别行政区""",2,"[""香港"", ""香港特别行政区""]",8,3,2022,2024


In [ ]:
# Check whether the first four digits of countyID can replace city.
county_code_base = (
    stkcd_info.with_columns(
        pl.col("countyID").cast(pl.Utf8).str.strip_chars().alias("countyID_str")
    )
    .with_columns(
        pl.when(pl.col("countyID_str").str.contains(r"^\d{6}$"))
        .then(pl.col("countyID_str").str.slice(0, 4))
        .otherwise(None)
        .alias("county_code4")
    )
)

county_code_valid = county_code_base.filter(
    pl.col("city").is_not_null() & pl.col("county_code4").is_not_null()
)

code4_to_city_conflicts = (
    county_code_valid.group_by("county_code4")
    .agg(
        pl.col("province").n_unique().alias("n_province"),
        pl.col("province").unique().sort().alias("province_values"),
        pl.col("city").n_unique().alias("n_city"),
        pl.col("city").unique().sort().alias("city_values"),
        pl.len().alias("n_rows"),
        pl.col("stkcd").n_unique().alias("n_stkcd"),
        pl.col("year").min().alias("first_year"),
        pl.col("year").max().alias("last_year"),
    )
    .filter(pl.col("n_city") > 1)
    .sort("county_code4")
)

code4_to_province_conflicts = (
    county_code_valid.group_by("county_code4")
    .agg(
        pl.col("province").n_unique().alias("n_province"),
        pl.col("province").unique().sort().alias("province_values"),
        pl.col("city").unique().sort().alias("city_values"),
        pl.len().alias("n_rows"),
        pl.col("stkcd").n_unique().alias("n_stkcd"),
    )
    .filter(pl.col("n_province") > 1)
    .sort("county_code4")
)

city_to_code4_conflicts = (
    county_code_valid.group_by("city")
    .agg(
        pl.col("county_code4").n_unique().alias("n_county_code4"),
        pl.col("county_code4").unique().sort().alias("county_code4_values"),
        pl.col("province").n_unique().alias("n_province"),
        pl.col("province").unique().sort().alias("province_values"),
        pl.len().alias("n_rows"),
        pl.col("stkcd").n_unique().alias("n_stkcd"),
        pl.col("year").min().alias("first_year"),
        pl.col("year").max().alias("last_year"),
    )
    .filter(pl.col("n_county_code4") > 1)
    .sort("city")
)

province_city_to_code4_conflicts = (
    county_code_valid.group_by(["province", "city"])
    .agg(
        pl.col("county_code4").n_unique().alias("n_county_code4"),
        pl.col("county_code4").unique().sort().alias("county_code4_values"),
        pl.len().alias("n_rows"),
        pl.col("stkcd").n_unique().alias("n_stkcd"),
        pl.col("year").min().alias("first_year"),
        pl.col("year").max().alias("last_year"),
    )
    .filter(pl.col("n_county_code4") > 1)
    .sort(["province", "city"])
)

county_code4_check_summary = pl.DataFrame(
    {
        "metric": [
            "rows",
            "missing_or_invalid_countyID",
            "unique_city",
            "unique_county_code4",
            "code4_to_multiple_city",
            "code4_to_multiple_province",
            "city_to_multiple_code4",
            "province_city_to_multiple_code4",
        ],
        "value": [
            county_code_base.height,
            county_code_base.filter(pl.col("county_code4").is_null()).height,
            county_code_valid.select(pl.col("city").n_unique()).item(),
            county_code_valid.select(pl.col("county_code4").n_unique()).item(),
            code4_to_city_conflicts.height,
            code4_to_province_conflicts.height,
            city_to_code4_conflicts.height,
            province_city_to_code4_conflicts.height,
        ],
    }
)

print(county_code4_check_summary)
print("Inspect code4_to_city_conflicts and city_to_code4_conflicts for exceptions.")
code4_to_city_conflicts

## Balanced city/industry aggregation summary


In [ ]:

from pathlib import Path

import polars as pl

PROJECT_ROOT = Path('/home/ubuntu/patent_similarity_new')
MODELS = ['minilm', 'distiluse']
AGGREGATION_OUTPUTS = {
    'city': {
        'entity_col': 'city_code',
        'dir': PROJECT_ROOT / 'output' / 'city_year_embeddings',
        'prefix': 'city_year',
    },
    'industry': {
        'entity_col': 'Ind',
        'dir': PROJECT_ROOT / 'output' / 'industry_year_embeddings',
        'prefix': 'industry_year',
    },
}


def embedding_columns(df: pl.DataFrame) -> list[str]:
    return [col for col in df.columns if col.startswith('emb_')]


def summarize_aggregation(level: str, model: str, weighted: bool = False) -> pl.DataFrame:
    config = AGGREGATION_OUTPUTS[level]
    prefix = config['prefix'] + ('_citweighted' if weighted else '')
    path = config['dir'] / f'{prefix}_{model}_embeddings.parquet'
    if not path.exists():
        return pl.DataFrame({
            'level': [level],
            'model': [model],
            'weighted': [weighted],
            'path': [str(path)],
            'exists': [False],
        })

    df = pl.read_parquet(path)
    emb_cols = embedding_columns(df)
    entity_col = config['entity_col']
    row_count = df.height
    n_entities = df.select(pl.col(entity_col).n_unique()).item()
    n_years = df.select(pl.col('p_year').n_unique()).item()
    expected_rows = n_entities * n_years
    zero_patents = df.filter(pl.col('n_patents') == 0).height
    zero_texts = df.filter(pl.col('n_texts_used') == 0).height
    null_embeddings = (
        df.filter(pl.all_horizontal([pl.col(col).is_null() for col in emb_cols])).height
        if emb_cols
        else None
    )
    return pl.DataFrame({
        'level': [level],
        'model': [model],
        'weighted': [weighted],
        'path': [str(path)],
        'exists': [True],
        'rows': [row_count],
        'entities': [n_entities],
        'years': [n_years],
        'expected_rows': [expected_rows],
        'is_balanced': [row_count == expected_rows],
        'zero_patent_rows': [zero_patents],
        'zero_patent_share': [zero_patents / row_count if row_count else None],
        'zero_text_rows': [zero_texts],
        'zero_text_share': [zero_texts / row_count if row_count else None],
        'null_embedding_rows': [null_embeddings],
        'null_embedding_share': [null_embeddings / row_count if row_count and null_embeddings is not None else None],
    })


aggregation_summary = pl.concat(
    [
        summarize_aggregation(level, model, weighted)
        for level in AGGREGATION_OUTPUTS
        for model in MODELS
        for weighted in [False, True]
    ],
    how='diagonal',
)
aggregation_summary


In [ ]:

def yearly_zero_patent_summary(level: str, model: str, weighted: bool = False) -> pl.DataFrame:
    config = AGGREGATION_OUTPUTS[level]
    prefix = config['prefix'] + ('_citweighted' if weighted else '')
    path = config['dir'] / f'{prefix}_{model}_embeddings.parquet'
    if not path.exists():
        return pl.DataFrame()
    return (
        pl.read_parquet(path)
        .group_by('p_year')
        .agg(
            pl.len().alias('rows'),
            (pl.col('n_patents') == 0).sum().alias('zero_patent_rows'),
            (pl.col('n_texts_used') == 0).sum().alias('zero_text_rows'),
        )
        .with_columns(
            pl.lit(level).alias('level'),
            pl.lit(model).alias('model'),
            pl.lit(weighted).alias('weighted'),
            (pl.col('zero_patent_rows') / pl.col('rows')).alias('zero_patent_share'),
            (pl.col('zero_text_rows') / pl.col('rows')).alias('zero_text_share'),
        )
        .select('level', 'model', 'weighted', 'p_year', 'rows', 'zero_patent_rows', 'zero_patent_share', 'zero_text_rows', 'zero_text_share')
        .sort(['level', 'model', 'weighted', 'p_year'])
    )


yearly_zero_patents = pl.concat(
    [
        yearly_zero_patent_summary(level, model, weighted)
        for level in AGGREGATION_OUTPUTS
        for model in MODELS
        for weighted in [False, True]
    ],
    how='diagonal',
)
yearly_zero_patents
